# 2. MITMA: Bronze → Silver

Esquema y descripción de las tablas: [silver.md](../docs/silver.md).

In [13]:
from pathlib import Path
import duckdb
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
DB = ROOT / 'data' / 'processed' / 'mitma.duckdb'
con = duckdb.connect(str(DB))
con.execute("SET memory_limit='80%'")
con.execute('CREATE SCHEMA IF NOT EXISTS silver')

In [ ]:
con.execute('LOAD spatial')
# Recrear únicamente las tablas generadas, primero las dependientes.
for tabla in ['geografia_zonas', 'viajes', 'personas', 'pernoctaciones', 'calidad_distritos', 'correspondencias', 'zonas']:
    con.execute(f'DROP TABLE IF EXISTS silver.{tabla}')

## Catálogo de zonas y auxiliares

In [5]:
con.execute("""CREATE TABLE IF NOT EXISTS silver.zonas (
    nivel VARCHAR,
    zona VARCHAR,
    nombre VARCHAR,
    poblacion_referencia DOUBLE,
    archivo_nombres VARCHAR,
    archivo_poblacion VARCHAR,
    en_catalogo BOOLEAN,
    PRIMARY KEY (nivel, zona)
)""")

con.execute("""INSERT INTO silver.zonas
WITH catalogo AS (    SELECT coalesce(n._nivel, p._nivel) AS nivel,
           coalesce(n.ID, p.column0) AS zona,
           nullif(n.name, '') AS nombre,
           cast(nullif(nullif(p.column1, ''), 'NA') AS DOUBLE) AS poblacion_referencia,
           n._archivo AS archivo_nombres, p._archivo AS archivo_poblacion
    FROM bronze.nombres_zonas n FULL JOIN bronze.poblacion_zonas p
    ON n._nivel=p._nivel AND n.ID=p.column0), codigos AS (SELECT _nivel AS nivel, ID AS zona FROM bronze.nombres_zonas
        UNION SELECT _nivel, column0 FROM bronze.poblacion_zonas
        UNION SELECT _nivel, ID FROM bronze.zonas_geograficas
        UNION SELECT _nivel, ID FROM bronze.centroides
        UNION SELECT _nivel, origen FROM bronze.viajes
        UNION SELECT _nivel, destino FROM bronze.viajes
        UNION SELECT _nivel, zona_pernoctacion FROM bronze.personas
        UNION SELECT _nivel, zona_residencia FROM bronze.pernoctaciones
        UNION SELECT _nivel, zona_pernoctacion FROM bronze.pernoctaciones
        UNION SELECT 'distritos', zona FROM bronze.calidad_distritos)
    SELECT c.nivel, c.zona, n.nombre, n.poblacion_referencia,
           n.archivo_nombres, n.archivo_poblacion, n.zona IS NOT NULL AS en_catalogo
    FROM codigos c LEFT JOIN catalogo n USING(nivel, zona)""")
con.execute("""CREATE TABLE IF NOT EXISTS silver.correspondencias (
    id_correspondencia BIGINT,
    seccion_ine VARCHAR,
    distrito_ine VARCHAR,
    municipio_ine VARCHAR,
    distrito_mitma VARCHAR,
    municipio_mitma VARCHAR,
    gau_mitma VARCHAR,
    archivo_origen VARCHAR,
    PRIMARY KEY (id_correspondencia)
)""")

con.execute("""INSERT INTO silver.correspondencias
SELECT row_number() OVER () AS id_correspondencia, nullif(nullif(seccion_ine, 'NA'), '') AS seccion_ine,
           nullif(nullif(distrito_ine, 'NA'), '') AS distrito_ine,
           nullif(nullif(municipio_ine, 'NA'), '') AS municipio_ine,
           distrito_mitma, municipio_mitma, gau_mitma, _archivo AS archivo_origen
    FROM bronze.correspondencias""")
con.execute("""CREATE TABLE IF NOT EXISTS silver.calidad_distritos (
    fecha DATE,
    nivel VARCHAR,
    zona VARCHAR,
    archivo_origen VARCHAR,
    PRIMARY KEY (fecha, nivel, zona),
    FOREIGN KEY (nivel, zona) REFERENCES silver.zonas (nivel, zona)
)""")

con.execute("""INSERT INTO silver.calidad_distritos
SELECT cast(strptime(fecha, '%Y%m%d') AS DATE) AS fecha,
           'distritos' AS nivel, zona, _archivo AS archivo_origen
    FROM bronze.calidad_distritos""")
display(con.sql('SELECT nivel, count(*) zonas, count(*) FILTER (WHERE nombre IS NULL) sin_nombre, count(*) FILTER (WHERE poblacion_referencia IS NULL) sin_poblacion FROM silver.zonas GROUP BY nivel').df())


,nivel,zonas,sin_nombre,sin_poblacion
0,GAU,4185,2099,2148
1,distritos,3910,118,167
2,municipios,4332,1714,1763


In [6]:
con.sql("""
SELECT *
FROM silver.zonas
WHERE en_catalogo=False
""").fetchdf()

,nivel,zona,nombre,poblacion_referencia,archivo_nombres,archivo_poblacion,en_catalogo
0,distritos,FRL04,None,NaN,None,None,False
1,distritos,PT187,None,NaN,None,None,False
2,distritos,FRD21,None,NaN,None,None,False
3,distritos,FRG02,None,NaN,None,None,False
4,distritos,FRC23,None,NaN,None,None,False
...,...,...,...,...,...,...,...
3926,municipios,FRJ24,None,NaN,None,None,False
3927,municipios,FRJ26,None,NaN,None,None,False
3928,municipios,PT185,None,NaN,None,None,False
3929,municipios,FRK12,None,NaN,None,None,False


## Viajes

In [7]:
con.execute("""CREATE TABLE IF NOT EXISTS silver.viajes (
    id_viaje BIGINT,
    nivel VARCHAR,
    fecha DATE,
    hora SMALLINT,
    origen VARCHAR,
    destino VARCHAR,
    distancia_rango VARCHAR,
    actividad_origen VARCHAR,
    actividad_destino VARCHAR,
    estudio_origen_posible VARCHAR,
    estudio_destino_posible VARCHAR,
    provincia_residencia VARCHAR,
    renta_rango VARCHAR,
    edad_rango VARCHAR,
    sexo VARCHAR,
    viajes DOUBLE,
    viajes_km DOUBLE,
    archivo_origen VARCHAR,
    PRIMARY KEY (id_viaje),
    FOREIGN KEY (nivel, origen) REFERENCES silver.zonas (nivel, zona),
    FOREIGN KEY (nivel, destino) REFERENCES silver.zonas (nivel, zona)
)""")

con.execute("""INSERT INTO silver.viajes
SELECT row_number() OVER () AS id_viaje, _nivel AS nivel, cast(strptime(fecha, '%Y%m%d') AS DATE) AS fecha,
           cast(periodo AS SMALLINT) AS hora, origen, destino,
           distancia AS distancia_rango, actividad_origen, actividad_destino,
           estudio_origen_posible, estudio_destino_posible,
           residencia AS provincia_residencia, renta AS renta_rango,
           nullif(nullif(edad, 'NA'), '') AS edad_rango,
           nullif(nullif(sexo, 'NA'), '') AS sexo,
           cast(nullif(nullif(viajes, 'NA'), '') AS DOUBLE) AS viajes,
           cast(nullif(nullif(viajes_km, 'NA'), '') AS DOUBLE) AS viajes_km,
           _archivo AS archivo_origen
    FROM bronze.viajes""")
display(con.sql('SELECT * FROM silver.viajes LIMIT 5').df())


,id_viaje,nivel,fecha,hora,origen,destino,distancia_rango,actividad_origen,actividad_destino,estudio_origen_posible,estudio_destino_posible,provincia_residencia,renta_rango,edad_rango,sexo,viajes,viajes_km,archivo_origen
0,1,distritos,2023-01-01,0,01001,01001,2-10,casa,frecuente,no,no,01,10-15,NaN,None,2.922,12.806,data/mitma/por-distritos/viajes/diarios/202301...
1,2,distritos,2023-01-01,0,01001,01001,2-10,frecuente,casa,no,no,01,10-15,NaN,None,5.202,22.098,data/mitma/por-distritos/viajes/diarios/202301...
2,3,distritos,2023-01-01,1,01001,01001,2-10,no_frecuente,no_frecuente,no,no,31,>15,25-45,None,4.648,20.805,data/mitma/por-distritos/viajes/diarios/202301...
3,4,distritos,2023-01-01,2,01001,01001,2-10,casa,frecuente,no,no,01,10-15,NaN,None,2.280,9.213,data/mitma/por-distritos/viajes/diarios/202301...
4,5,distritos,2023-01-01,3,01001,01001,2-10,frecuente,casa,no,no,01,10-15,NaN,None,2.922,11.720,data/mitma/por-distritos/viajes/diarios/202301...


## Personas y pernoctaciones

In [8]:
con.execute("""CREATE TABLE IF NOT EXISTS silver.personas (
    id_personas BIGINT,
    nivel VARCHAR,
    fecha DATE,
    zona_pernoctacion VARCHAR,
    edad_rango VARCHAR,
    sexo VARCHAR,
    numero_viajes VARCHAR,
    personas DOUBLE,
    archivo_origen VARCHAR,
    PRIMARY KEY (id_personas),
    FOREIGN KEY (nivel, zona_pernoctacion) REFERENCES silver.zonas (nivel, zona)
)""")

con.execute("""INSERT INTO silver.personas
SELECT row_number() OVER () AS id_personas, _nivel AS nivel, cast(strptime(fecha, '%Y%m%d') AS DATE) AS fecha,
           zona_pernoctacion, nullif(nullif(edad, 'NA'), '') AS edad_rango,
           nullif(nullif(sexo, 'NA'), '') AS sexo, numero_viajes,
           cast(nullif(nullif(personas, 'NA'), '') AS DOUBLE) AS personas,
           _archivo AS archivo_origen
    FROM bronze.personas""")
con.execute("""CREATE TABLE IF NOT EXISTS silver.pernoctaciones (
    nivel VARCHAR,
    fecha DATE,
    zona_residencia VARCHAR,
    zona_pernoctacion VARCHAR,
    personas DOUBLE,
    archivo_origen VARCHAR,
    PRIMARY KEY (nivel, fecha, zona_residencia, zona_pernoctacion),
    FOREIGN KEY (nivel, zona_residencia) REFERENCES silver.zonas (nivel, zona),
    FOREIGN KEY (nivel, zona_pernoctacion) REFERENCES silver.zonas (nivel, zona)
)""")

con.execute("""INSERT INTO silver.pernoctaciones
SELECT _nivel AS nivel, cast(strptime(fecha, '%Y%m%d') AS DATE) AS fecha,
           zona_residencia, zona_pernoctacion,
           cast(nullif(nullif(personas, 'NA'), '') AS DOUBLE) AS personas,
           _archivo AS archivo_origen
    FROM bronze.pernoctaciones""")
for tabla in ['personas', 'pernoctaciones']:
    display(con.sql(f'SELECT * FROM silver.{tabla} LIMIT 5').df())


,id_personas,nivel,fecha,zona_pernoctacion,edad_rango,sexo,numero_viajes,personas,archivo_origen
0,1,distritos,2023-01-01,01001,0-25,hombre,0,251.165,data/mitma/por-distritos/personas/diarios/2023...
1,2,distritos,2023-01-01,01001,0-25,hombre,1,53.172,data/mitma/por-distritos/personas/diarios/2023...
2,3,distritos,2023-01-01,01001,0-25,hombre,2,59.478,data/mitma/por-distritos/personas/diarios/2023...
3,4,distritos,2023-01-01,01001,0-25,hombre,2+,96.707,data/mitma/por-distritos/personas/diarios/2023...
4,5,distritos,2023-01-01,01001,0-25,mujer,0,201.435,data/mitma/por-distritos/personas/diarios/2023...


,nivel,fecha,zona_residencia,zona_pernoctacion,personas,archivo_origen
0,distritos,2023-01-01,01001,01001,2329.644,data/mitma/por-distritos/pernoctaciones/diario...
1,distritos,2023-01-01,01001,01017_AM,16.682,data/mitma/por-distritos/pernoctaciones/diario...
2,distritos,2023-01-01,01001,01028_AM,2.922,data/mitma/por-distritos/pernoctaciones/diario...
3,distritos,2023-01-01,01001,01051,6.056,data/mitma/por-distritos/pernoctaciones/diario...
4,distritos,2023-01-01,01001,01054_AM,5.897,data/mitma/por-distritos/pernoctaciones/diario...


## Geografía

In [9]:
con.execute("""CREATE TABLE IF NOT EXISTS silver.geografia_zonas (
    nivel VARCHAR,
    zona VARCHAR,
    geometria GEOMETRY('EPSG:4326'),
    geometria_valida BOOLEAN,
    centroide GEOMETRY('EPSG:4326'),
    archivo_poligono VARCHAR,
    archivo_centroide VARCHAR,
    PRIMARY KEY (nivel, zona),
    FOREIGN KEY (nivel, zona) REFERENCES silver.zonas (nivel, zona)
)""")

con.execute("""INSERT INTO silver.geografia_zonas
SELECT coalesce(p._nivel, c._nivel) AS nivel, coalesce(p.ID, c.ID) AS zona,
           ST_Transform(p.geom, 'EPSG:4326', always_xy := true) AS geometria,
           ST_IsValid(ST_Transform(p.geom, 'EPSG:4326', always_xy := true)) AS geometria_valida,
           ST_Transform(c.geom, 'EPSG:4326', always_xy := true) AS centroide,
           p._archivo AS archivo_poligono, c._archivo AS archivo_centroide
    FROM bronze.zonas_geograficas p FULL JOIN bronze.centroides c
      ON p._nivel=c._nivel AND p.ID=c.ID""")
control = con.sql("""SELECT nivel, count(*) AS zonas,
    count(*) FILTER (WHERE geometria IS NOT NULL AND NOT ST_IsValid(geometria)) AS invalidas,
    count(*) FILTER (WHERE geometria IS NULL) AS sin_poligono,
    count(*) FILTER (WHERE centroide IS NULL) AS sin_centroide
    FROM silver.geografia_zonas GROUP BY nivel""").df()
display(control)


,nivel,zonas,invalidas,sin_poligono,sin_centroide
0,distritos,3909,56,0,0
1,GAU,2203,54,0,0
2,municipios,2735,59,0,0


## Decisiones para Gold
- Agrupar viajes por nivel, fecha, hora, origen y destino para obtener flujos horarios. Mantener los niveles separados al sumar.
- A partir de esos flujos, calcular los viajes diarios emitidos y recibidos por zona.
- Sumar todas las filas conservadas en Silver, incluidas las que repiten desgloses, sin deduplicarlas ni aplicar filtros de calidad automáticos. La lista de distritos descartados queda como auxiliar; no estar en ella no garantiza que una zona no tenga incidencias.
- Acompañar los totales con contadores de medidas nulas. No inventar ceros para combinaciones ausentes. Personas y pernoctaciones se mantienen en Silver.

In [10]:
con.sql("SHOW TABLES")

nameVARCHAR


In [11]:
con.close()
print('Silver finished.')

Silver finished.
